# Homework 2 - Xinlei Cheng

In [0]:
from pyspark.sql.functions import (
    col, avg, min, max, rank, row_number, count, sum,
    when, current_date, floor, months_between, datediff,
    to_date, concat, lit, upper, substring, coalesce,
    first, last, dense_rank, struct
)
from pyspark.sql.window import Window

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

### Question 1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

Logic: The `pit_stops` dataset contains one row per pit stop event. Each row has `raceId`, `driverId`, and `milliseconds` (the duration of that stop). A driver can make multiple pit stops per race, so:

1. I group by `raceId` and `driverId` and compute the average pit stop duration per driver per race.
2. I group by `raceId` alone and compute the **minimum** (fastest) and **maximum** (slowest) individual pit stop across all drivers in each race.
3. I join the two results together so each row shows the driver's average alongside with the fastest and slowest results.

I use `milliseconds` (cast to integer) because it is the most precise numeric measure of pit stop duration.

In [0]:
df_pit_stops.head(10)

In [0]:
df_pit = df_pit_stops.withColumn("milliseconds", col("milliseconds").cast("int"))

# Average pit stop time per driver per race
df_avg_pit_per_driver = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(
        avg("milliseconds").alias("avg_pit_time_ms")
    )
)

display(df_avg_pit_per_driver)


In [0]:
# Fastest and slowest pit stop per race
df_race_pit_extremes = (
    df_pit
    .groupBy("raceId")
    .agg(
        min(col("milliseconds")) .alias("fastest_pit_stop_ms"),
        max(col("milliseconds")).alias("slowest_pit_stop_ms")
    )
)

# Combine them into one
df_q1 = (
    df_avg_pit_per_driver
    .join(df_race_pit_extremes, on="raceId", how="inner")
    .orderBy("raceId", "driverId")
)

display(df_q1)

### Code Explanation

1. **`.withColumn("milliseconds", col("milliseconds").cast("int"))`** — The CSV reader loads all columns as strings. This casts `milliseconds` to an integer so that `avg`, `min`, and `max` perform numeric operations.

2. **`.groupBy("raceId", "driverId").agg(avg("milliseconds").alias("avg_pit_time_ms"))`** — Groups pit stop records by race and driver, then computes the mean duration. `.alias()` renames the result column for clarification.

3. **`.groupBy("raceId").agg(min(...), max(...))`** — Groups at the race level to find the single fastest and slowest pit stop across all drivers and all stops in that race.

4. **`.join(..., on="raceId", how="inner")`** — Joins the per-driver averages.

5. **`.orderBy("raceId", "driverId")`** — Sorts for readability.

### Question 2：Rank order by finishing position the average time spent at the pit stop in each race.

Logic: This question asks: within each race, rank drivers by their finishing position and show their average pit stop time. The goal is to see whether drivers who finish higher tend to have faster pit stops.

1. Compute the average pit stop time per driver per race from `pit_stops`.
2. Get finishing positions from `results` (using `positionOrder`, which is always a clean integer — unlike `position` which can contain `\\N` for DNFs).
3. Join the two on `raceId` and `driverId`.
4. Order by `raceId` and `positionOrder` so that within each race, drivers are listed from 1st place downward.

In [0]:
# reuse df_pit from last question
df_avg_pit = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(avg("milliseconds").alias("avg_pit_time_ms"))
)

# Get finishing positions from results
df_finish = (
    df_results
    .select(
        col("raceId"),
        col("driverId"),
        col("positionOrder").cast("int").alias("finishing_position")
    )
)

# Join and order by finishing position within each race
df_q2 = (
    df_finish
    .join(df_avg_pit, on=["raceId", "driverId"], how="inner")
    .orderBy(col("raceId"), col("finishing_position"))
)

display(df_q2)

### Code Explanation

1. **`df_avg_pit`** — Groups `pit_stops` by `raceId` and `driverId` and computes the mean pit stop duration in milliseconds. This is the same logic as Q1.

2. **`df_finish`** — Selects `raceId`, `driverId`, and `positionOrder` from the `results` table. I use `positionOrder` rather than `position` because `positionOrder` is always a clean integer (1, 2, 3, ...) even for drivers who did not finish (DNF), while `position` can contain `\\N`. The `.cast("int")` converts the string to an integer for proper numeric sorting.

3. **`.join(..., on=["raceId", "driverId"], how="inner")`** — An inner join keeps only drivers who appear in both datasets (i.e., drivers who made at least one pit stop and have a race result). Drivers who never pitted (e.g., some DNFs on lap 1) are excluded.

4. **`.orderBy(col("raceId"), col("finishing_position"))`** — Sorts by race, then by finishing position within each race, so the output reads as a rank-ordered list.

### Extra Credit: Alternative Approach Using `rank()` Window Function

We can explicitly rank drivers by their average pit stop time within each race using the `rank()` window function, and then compare that pit stop ranking to their finishing position.

In [0]:
# Alternative: add an explicit pit stop rank column for comparison
pit_rank_window = Window.partitionBy("raceId").orderBy("avg_pit_time_ms")

df_q2_alt = (
    df_q2
    .withColumn("pit_stop_rank", rank().over(pit_rank_window))
    .orderBy(col("raceId"), col("finishing_position"))
)

display(df_q2_alt)

**Alternative explanation:** `Window.partitionBy("raceId").orderBy("avg_pit_time_ms")` creates a window partitioned by race and ordered by average pit time (ascending, so the fastest pit strategy gets rank 1). The `rank()` function assigns a rank within each race. This lets us directly compare `finishing_position` and `pit_stop_rank` — if they correlate, it suggests pit stop speed contributes to race outcomes.

### Question 3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

Logic: The `drivers` dataset has a `code` column representing a 3-letter abbreviation (e.g., "HAM" for Hamilton, "ALO" for Alonso). Some drivers have missing codes — stored as `\\N` in the CSV.

My approach to generate the missing codes:

1. Identify rows where `code` is `\\N` (missing).
2. For those drivers, generate a code using the first three letters of the driver's surname, converted to uppercase. This mirrors the convention used by the FIA for most modern drivers.
3. Use `when().otherwise()` to conditionally replace only the missing values, leaving existing valid codes untouched.
4. Verify there are no remaining missing codes.

In [0]:
# Check how many drivers have missing codes
print("Drivers with missing code (\\N):")
df_drivers.filter(col("code") == "\\N").select("driverId", "driverRef", "forename", "surname", "code").show(20, truncate=False)

In [0]:
# Fill missing codes with first 3 letters of surname
df_drivers_fixed = df_drivers.withColumn(
    "code",
    when(
        (col("code") == "\\N") | (col("code").isNull()),
        upper(substring(col("surname"), 1, 3))
    ).otherwise(col("code"))
)

display(df_drivers_fixed.select("driverId", "forename", "surname", "code"))

In [0]:
# Verify no missing codes remain
missing_count = df_drivers_fixed.filter((col("code") == "\\N") | (col("code").isNull())).count()
print(f"Remaining missing codes: {missing_count}")

### Code Explanation

1. **`.filter(col("code") == "\\N")`** — Identifies drivers whose `code` field is the placeholder string `\\N`, which represents missing data in this CSV format.

2. **`when((col("code") == "\\N") | (col("code").isNull()), ...).otherwise(col("code"))`** — A conditional expression: if the code is `\\N` or null, apply the replacement logic; otherwise, keep the existing code. This ensures we only modify rows that actually need fixing.

3. **`upper(substring(col("surname"), 1, 3))`** — `substring(col("surname"), 1, 3)` extracts the first 3 characters of the surname (1-indexed). `upper()` converts them to uppercase. For example, "Alonso" becomes "ALO", "Schumacher" becomes "SCH".

4. **The verification step** counts any remaining `\\N` or null codes to confirm all have been filled.


### Question 4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

Logic: To determine each driver's age at the time of a race:

1. Join `results` with `drivers` (to get `dob`) and `races` (to get the race `date`).
2. Compute **Age** as the number of complete years between the driver's date of birth (`dob`) and the race date. I use `months_between()` divided by 12 and then `floor()` to get the age number. 
3. Within each race, identify the youngest (minimum age) and oldest (maximum age) driver.


In [0]:
# Join results with drivers and races to get dob and race date
df_age = (
    df_results
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="inner")
    .join(df_races.select(col("raceId"), col("name").alias("race_name"), col("date").alias("race_date")), on="raceId", how="inner")
)

# Compute age in full years
df_age = df_age.withColumn(
    "Age",
    floor(months_between(to_date(col("race_date")), to_date(col("dob"))) / 12).cast("int")
)

display(df_age.select("raceId", "race_name", "driverId", "forename", "surname", "dob", "race_date", "Age").orderBy("raceId", "Age"))

In [0]:
# Find youngest and oldest driver in each race
youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

df_youngest = (
    df_age
    .withColumn("rn", row_number().over(youngest_window))
    .filter(col("rn") == 1)
    .select(
        col("raceId"), col("race_name"),
        col("forename").alias("youngest_forename"),
        col("surname").alias("youngest_surname"),
        col("Age").alias("youngest_age")
    )
)

df_oldest = (
    df_age
    .withColumn("rn", row_number().over(oldest_window))
    .filter(col("rn") == 1)
    .select(
        col("raceId"),
        col("forename").alias("oldest_forename"),
        col("surname").alias("oldest_surname"),
        col("Age").alias("oldest_age")
    )
)

df_q4 = df_youngest.join(df_oldest, on="raceId", how="inner").orderBy("raceId")

display(df_q4)

### Code Explanation

1. **Two joins** connect `results` to `drivers` (to get dob) and to `races` (to get `race_date`) using their shared keys. I alias `name` as `race_name` and `date` as `race_date` to avoid column name conflicts.

2. **`floor(months_between(to_date(col("race_date")), to_date(col("dob"))) / 12)`** — `to_date()` converts the string columns to proper date types. `months_between()` computes the exact number of months between the race date and the birth date. Dividing by 12 converts to years, and `floor()` rounds down to the nearest integer, giving the driver's age in complete years.

3. **`Window.partitionBy("raceId").orderBy(col("Age").asc())`** — Creates a window per race, ordered by age ascending. `row_number()` over this window assigns rank 1 to the youngest driver. The same logic (with `.desc()`) finds the oldest.

4. **`.filter(col("rn") == 1)`** — Keeps only the youngest or oldest per race

5. The final join combines the youngest and oldest into a single row per race for easier comparison

### Question 5: At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver.

## Logic: This question asks for a cumulative count

1. Join `results` with `races` to get the race `date` so we can have an chronological order.
2. Create indicator columns: `is_win` (1 if `positionOrder` == 1), `is_2nd` (1 if == 2), `is_3rd` (1 if == 3), otherwise 0.
3. Use a cumulative window partitioned by `driverId` and ordered by `race_date` (and `raceId` as tiebreaker) to compute running sums of each indicator.
4. The result shows, for each driver at each race, their career podium tallies up to that point.

In [0]:
# Join results with race
df_podium = (
    df_results
    .join(
        df_races.select(col("raceId"), col("date").alias("race_date"), col("name").alias("race_name")),
        on="raceId",
        how="inner"
    )
    .withColumn("positionOrder", col("positionOrder").cast("int"))
)

# Create indicator columns for podium positions
df_podium = (
    df_podium
    .withColumn("is_win", when(col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_2nd", when(col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_3rd", when(col("positionOrder") == 3, 1).otherwise(0))
)

#Cumulative window
cum_window = (
    Window
    .partitionBy("driverId")
    .orderBy(to_date(col("race_date")), col("raceId"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_q5 = (
    df_podium
    .withColumn("cumulative_wins", sum("is_win").over(cum_window))
    .withColumn("cumulative_2nd_places", sum("is_2nd").over(cum_window))
    .withColumn("cumulative_3rd_places", sum("is_3rd").over(cum_window))
    .select(
        "raceId", "race_name", "race_date", "driverId",
        "positionOrder",
        "cumulative_wins", "cumulative_2nd_places", "cumulative_3rd_places"
    )
    .orderBy("driverId", to_date(col("race_date")))
)

display(df_q5)

### Code Explanation

1. **Join `results` with `races`** — We need `race_date` to get chronological ordering. I renamed columns to avoid naming conflicts.

2. **`when(col("positionOrder") == 1, 1).otherwise(0)`** — Creates binary indicator columns. `is_win` is 1 when the driver won (position 1), 0 otherwise. Same logic for 2nd and 3rd place.

3. **`Window.partitionBy("driverId").orderBy(...).rowsBetween(Window.unboundedPreceding, Window.currentRow)`** — This defines a cumulative window for each driver. It starts from the very first race the driver participated in (`unboundedPreceding`) up to and including the current race (`currentRow`). The ordering is by `race_date` with `raceId` as a tiebreaker.

4. **`sum("is_win").over(cum_window)`** — Computes the running total of wins for each driver across all races up to the current one. The same pattern applies for 2nd and 3rd place counts.

5. The final **`.select()`** keeps only the relevant columns, and **`.orderBy("driverId", ...)`** sorts by driver and chronologically for readability.

### Question 6: Continue exploring the data by answering your own question.

My question is: Which team has the best average finishing position in each season, and how has constructor dominance evolved over time?

Logic: I want to explore team dominance across F1 seasons:

1. Join `results` with `races` to get the `year` (season) for each result.
2. Group by `year` and `constructorId` to compute each constructor's **average finishing position** across all race entries in that season (using `positionOrder`, where lower = better).
3. Rank constructors within each season by their average finishing position.
4. Filter for the top constructor (rank 1) per season to show who dominated each year.

This reveals patterns of dominance. For example, long periods where one team consistently outperformed the rest.

In [0]:
# Join results with races to get season year
df_constructor = (
    df_results
    .join(
        df_races.select(col("raceId"), col("year")),
        on="raceId", how="inner"
    )
    .withColumn("positionOrder", col("positionOrder").cast("int"))
)

# Average finishing position per team per season
df_constructor_avg = (
    df_constructor
    .groupBy("year", "constructorId")
    .agg(
        avg("positionOrder").alias("avg_finishing_position"),
        count("*").alias("total_entries")
    )
)

# Rank teams within each season
season_window = Window.partitionBy("year").orderBy(col("avg_finishing_position").asc())

df_constructor_ranked = (
    df_constructor_avg
    .withColumn("season_rank", rank().over(season_window))
    .orderBy(col("year"), col("season_rank"))
)

display(df_constructor_ranked)

### Code Explanation

1. **Join `results` with `races`**: Brings in the year column so we can group results by season.

2. **`.groupBy("year", "constructorId").agg(avg("positionOrder"), count("*"))`** :For each team in each year, computes the average finishing position and the total number of race entries.

3. **`Window.partitionBy("year").orderBy(col("avg_finishing_position").asc())`** : Defines a window per season, ordered by average finishing position ascending. The best performing team gets rank 1.

4. **`rank().over(season_window)`**: Assigns a rank to each team within each season based on their average finishing position.

5. **`.filter(col("season_rank") == 1)`** — Isolates just the dominant team per season, providing a clean timeline of F1 team dominance.